# Lecture 4: Algerian Forest Fires Project - Data Setup and Cleaning

### Short, beginner-friendly project notes

This notebook starts a real Ridge, Lasso, and Elastic Net project using the Algerian Forest Fires dataset. We focus on understanding the data, cleaning it safely, and preparing it for later modelling.

**Main idea:** Good models start with clean, correctly typed data. Do not rush to train before checking the dataset.

## 1. Project goal

The project workflow is:

1. Understand the dataset.
2. Clean missing, repeated, and wrongly typed values.
3. Do exploratory data analysis (EDA).
4. Create or improve useful features.
5. Train Linear Regression, Ridge, Lasso, and Elastic Net.
6. Tune model settings with cross-validation and evaluate on a final test set.

For this regression project, we will use **FWI (Fire Weather Index)** as the target. The remaining weather and fire-index values can be used as input features.

## 2. Dataset in simple words

The original dataset contains records from two Algerian regions in June to September 2012: Bejaia and Sidi Bel-Abbes.

The raw source has 244 records. The cleaned CSV included with this project has **243 usable rows** after removing separator/header rows and missing data.

| Column | Meaning |
|---|---|
| Temperature | Temperature in degrees Celsius |
| RH | Relative humidity |
| Ws | Wind speed |
| Rain | Rainfall |
| FFMC, DMC, DC, ISI, BUI | Fire-weather system indexes |
| FWI | Fire Weather Index - our regression target |
| Classes | fire or not fire - useful for a classification version |
| Region | 0 for Bejaia and 1 for Sidi Bel-Abbes |

The same data can support two tasks: predict the numeric FWI value (regression) or predict fire/not fire (classification).

## 3. Safe data workflow

The diagram below shows the order to follow. Every later model depends on the earlier cleaning steps being correct.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

steps = [
    ("Load CSV", "Read the file"),
    ("Inspect", "shape, columns, dtypes"),
    ("Clean", "missing rows and spaces"),
    ("Convert", "numbers to numeric types"),
    ("Explore", "plots and summaries"),
    ("Model later", "split, scale, train"),
]

fig, ax = plt.subplots(figsize=(14, 3))
ax.set_xlim(0, 12)
ax.set_ylim(0, 2)
ax.axis("off")

for index, (title, detail) in enumerate(steps):
    x = 0.15 + index * 1.95
    box = FancyBboxPatch((x, 0.7), 1.55, 0.7, boxstyle="round,pad=0.03",
                         facecolor="#d9eaf7", edgecolor="#2f5d8a")
    ax.add_patch(box)
    ax.text(x + 0.775, 1.13, title, ha="center", va="center", weight="bold")
    ax.text(x + 0.775, 0.91, detail, ha="center", va="center", fontsize=8)
    if index < len(steps) - 1:
        ax.add_patch(FancyArrowPatch((x + 1.58, 1.05), (x + 1.88, 1.05),
                                     arrowstyle="->", mutation_scale=14, color="#2f5d8a"))

ax.set_title("A clean workflow for a machine-learning project", weight="bold")
plt.show()

## 4. Load the cleaned dataset

The practical files are kept in a subfolder. We use the cleaned CSV because it already removes the repeated regional header row found inside the raw source file.

We still strip spaces from class labels. This is important because a label with an extra space should not become a separate category.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

data_path = Path("Ridge Lassso Elastic Regression Practicals") / "Algerian_forest_fires_cleaned_dataset.csv"
df = pd.read_csv(data_path)
df["Classes"] = df["Classes"].str.strip()

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

### What to check first

- **Shape:** number of rows and columns.
- **Columns:** whether names match your expectation.
- **Head:** a quick preview to catch obvious mistakes.
- **Data types:** numeric columns should be integers or floats, not text objects.
- **Missing values:** a model cannot learn from a blank value without a clear plan.

In [ ]:
quality_check = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(),
})

quality_check

## 5. Important cleaning lessons from the raw file

The raw CSV is more difficult than the cleaned CSV. It contains a repeated header/separator line between the two regions and some missing values. A careful cleaning routine should:

1. Read the correct header row.
2. Add a region column before removing the separator row.
3. Remove missing or repeated-header rows.
4. Reset the index after dropping rows.
5. Strip spaces from column names and class labels.
6. Convert weather and index columns to numeric types.

**Why reset the index?** After deleting rows, the old row numbers may have gaps. A reset index gives the cleaned table neat row positions again.

## 6. Small EDA: class counts and FWI relationship

EDA means using summaries and plots to understand the data before modelling. The left plot counts fire and not-fire rows. The right plot shows the relationship between temperature and FWI, with color showing the region.

A plot can suggest a pattern, but it does not prove that one feature alone causes another.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["Classes"].value_counts().plot(kind="bar", ax=axes[0], color=["#e76f51", "#457b9d"], rot=0)
axes[0].set_title("Fire and not-fire records")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Number of rows")

scatter = axes[1].scatter(df["Temperature"], df["FWI"], c=df["Region"], cmap="coolwarm", alpha=0.75)
axes[1].set_title("Temperature versus FWI")
axes[1].set_xlabel("Temperature (°C)")
axes[1].set_ylabel("FWI")
legend = axes[1].legend(*scatter.legend_elements(), title="Region")
axes[1].add_artist(legend)

fig.suptitle("First look at the Algerian Forest Fires data", weight="bold")
fig.tight_layout()
plt.show()

## 7. Prepare features for the later regression model

For our future FWI regression model, X contains the input features and y contains the numeric target, FWI. We leave out Classes because it is the target for a different classification version of the project.

We will later split X and y into training and test data. Scaling must be learned from the training set only, preferably inside a pipeline.

In [ ]:
feature_columns = [
    "day", "month", "year", "Temperature", "RH", "Ws", "Rain",
    "FFMC", "DMC", "DC", "ISI", "BUI", "Region",
]
target_column = "FWI"

X = df[feature_columns]
y = df[target_column]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Any missing values in X?", X.isna().any().any())
X.head()

## 8. Common beginner mistakes to avoid

- Training before checking data types and missing values.
- Accidentally keeping a repeated header row as data.
- Treating class labels with and without spaces as different values.
- Scaling the full dataset before splitting it. This leaks information from test data.
- Using FWI as both an input and the target. Remove the target from X.
- Using Classes in an FWI regression model without deciding why it belongs there.

A good habit is to write one short note beside each cleaning decision: what changed, why it changed, and whether it may affect the final model.

## 9. Final revision card

- This project uses Algerian forest-fire weather data from two regions.
- The cleaned file has 243 usable records and 15 columns.
- FWI is the numeric regression target for Ridge, Lasso, and Elastic Net.
- Classes is useful for a separate fire/not-fire classification task.
- Clean data before modelling: fix headers, missing values, spaces, indices, and data types.
- EDA helps you understand distributions and possible relationships before fitting a model.
- Later: split data, scale inside a pipeline, tune models with cross-validation, then test once.

### One-line interview answer

**Before training a regression model, I inspect the dataset, clean missing and wrongly typed values, prevent data leakage, and clearly separate features from the target.**